# Process the MUDI dataset

https://zenodo.org/records/15544552

In [4]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 53.0 MB/s eta 0:00:00


In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import pandas as pd
import numpy as np
import json
import os
from rdkit import Chem
from rdkit.Chem import PandasTools

In [24]:
data_path = '/content/drive/MyDrive/FYP/IRP/Data/MUDI'
train_df = pd.read_csv(os.path.join(data_path, 'dataset/train.csv'))
test_df = pd.read_csv(os.path.join(data_path, 'dataset/test.csv'))


In [27]:
len(train_df['Drug1'].unique())

989

In [28]:
import json

# Load the JSON file
with open('/content/drive/MyDrive/FYP/IRP/Data/MUDI/drug_info.json', 'r') as f:
    drug_info = json.load(f)

# Number of unique drug IDs (keys of the dictionary)
num_unique_drugs = len(drug_info)
print(f"Number of unique drug IDs in drug_info.json: {num_unique_drugs}")

Number of unique drug IDs in drug_info.json: 1295


## Create a Mapping from Drug ID to SMILES

The drug IDs in the CSV files have the format "Compound::DBxxxxx", while the keys in drug_info are simply "DBxxxxx". Need to strip the prefix.

In [19]:
# Function to clean drug ID
def clean_drug_id(drug_id):
    # Remove 'Compound::' prefix if present
    return drug_id.replace('Compound::', '')

# Build a dictionary that maps cleaned ID to SMILES
id_to_smiles = {}
for drug_id, info in drug_info.items():
    if 'smiles' in info:
        id_to_smiles[drug_id] = info['smiles']

print("Example mapping:")
print("DB00842 ->", id_to_smiles.get('DB00842', 'Not found'))

Example mapping:
DB00842 -> C1=CC=C(C=C1)C2=NC(C(=O)NC3=C2C=C(C=C3)Cl)O


## Map SMILES to Each Row

Create new columns smiles_a and smiles_b by applying the cleaning and mapping.

In [20]:
def get_smiles(row, drug_col):
    raw_id = row[drug_col]
    clean_id = clean_drug_id(raw_id)
    return id_to_smiles.get(clean_id, None)

# Apply to both dataframes
for df in [train_df, test_df]:
    df['smiles_a'] = df.apply(lambda row: get_smiles(row, 'Drug1'), axis=1)
    df['smiles_b'] = df.apply(lambda row: get_smiles(row, 'Drug2'), axis=1)

# Check for missing SMILES (should be none if all drugs are in drug_info)
print("Missing SMILES in train:", train_df['smiles_a'].isna().sum() + train_df['smiles_b'].isna().sum())
print("Missing SMILES in test:", test_df['smiles_a'].isna().sum() + test_df['smiles_b'].isna().sum())

Missing SMILES in train: 0
Missing SMILES in test: 0


## Encode Interaction Labels


The dataset contains three interaction types:

- Synergism

- Antagonism

- New Effect (New Adverse)

Will map them to numeric values for classification.

In [21]:
# Define mapping
interaction_map = {
    'Synergism': 0,
    'Antagonism': 1,
    'New Effect (New Adverse)': 2
}

# Apply mapping
train_df['label'] = train_df['Interaction'].map(interaction_map)
test_df['label'] = test_df['Interaction'].map(interaction_map)

# Verify distribution
print("Train label distribution:\n", train_df['label'].value_counts())
print("\nTest label distribution:\n", test_df['label'].value_counts())

Train label distribution:
 label
0.0    186268
1.0     27320
Name: count, dtype: int64

Test label distribution:
 label
0.0    71001
1.0    11914
Name: count, dtype: int64


In [22]:
# Select and save train data
train_processed = train_df[['smiles_a', 'smiles_b', 'label']].copy()
train_processed.to_csv(os.path.join(data_path, 'train_processed.csv'), index=False)

# Select and save test data
test_processed = test_df[['smiles_a', 'smiles_b', 'label']].copy()
test_processed.to_csv(os.path.join(data_path, 'test_processed.csv'), index=False)

print("Processed files saved:")
print("- train_processed.csv")
print("- test_processed.csv")

Processed files saved:
- train_processed.csv
- test_processed.csv


In [23]:
train_processed.head ()

,smiles_a,smiles_b,label
0,C1=CC=C(C=C1)C2=NC(C(=O)NC3=C2C=C(C=C3)Cl)O,CC(=O)NC1=CC=C(C=C1)O,0.0
1,CC(C1=CC(=C(C=C1)C2=CC=CC=C2)F)C(=O)O,CC1=C(SC=N1)/C=C\C2=C(N3[C@@H]([C@@H](C3=O)NC(...,0.0
2,C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2C(=O)NC4=...,C1CC[C@H]([C@@H](C1)CN2CCN(CC2)C3=NSC4=CC=CC=C...,0.0
3,C[C@@H](CC1=CC=CC=C1)NC(=O)[C@H](CCCCN)N,CN1C(=C(C2=C(S1(=O)=O)C=CS2)O)C(=O)NC3=CC=CC=N3,0.0
4,CC1=CN(C(=O)NC1=O)[C@H]2C[C@@H]([C@H](O2)CO)N=...,CNC(=O)C1=NC=CC(=C1)OC2=CC=C(C=C2)NC(=O)NC3=CC...,0.0
